# Dans ce notebook nous allons commencer par chercher le jeu de données et le separer en entraînement/validation/test, nous allons aussi faire l'augmentation du jeu de donnée (avec des rotations principalement)
# Puis nous allons coder le U-Net.
Le code source est dans `glacier.model` :
- `dataset.py`
- `cnn_unet.py`
- `lost_functions.py`
- `training.py`

## Le test set sera isolé et ne devrait jamais être évalué dans ce notebook. On en fera une évaluation que à la toute fin avec notre modèle final.

In [1]:
from pathlib import Path
import json 
import numpy as np
import torch
from torch.utils.data import DataLoader

from glacier.data.data_fetching import repo_root
from glacier.model import (
    GlacierDataset,
    UNet,
    DiceBCELoss,
    compute_band_stats,
    discover_pairs,
    train_one_epoch,
    evaluate,
    plot_history,
    show_predictions,
)

ROOT = repo_root()

KeyboardInterrupt: 

# Hyperparamètres
## Attention ici, changer les hyperparamètres si c'est trop demandant (pourrait faire crash votre ordi)

In [ ]:
SEED          = 42
BATCH_SIZE    = 4           # HPC avec GPU : 16 ou 32
EPOCHS        = 100         # HPC : 200+ si le dataset grandit
LR            = 1e-3
WEIGHT_DECAY  = 1e-4
PAD_TO        = 320         # multiple de 16 pour les 4 maxpools du U-Net
N_BANDS       = 5           # blue, green, red, nir, swir16
TRAIN_REGION  = "caucasus"
EVAL_REGIONS  = ["alps", "andes"]
VAL_FRAC      = 0.50        # 50 % val, 50 % test (parmi alps+andes)

COMPOSITES_ROOT = ROOT / "data" / "sentinel2" / "composites"
MASKS_ROOT      = ROOT / "data" / "sentinel2" / "glims_masks"
CHECKPOINT_DIR  = ROOT / "data" / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
SPLITS_PATH = CHECKPOINT_DIR / "splits.json"

# CPU local (par défaut)
DEVICE = torch.device("cpu")

# HPC : décommenter la ligne suivante pour utiliser le GPU
# DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)

torch.manual_seed(SEED)
np.random.seed(SEED)

# Création du dataset en paire et split train/val/test

In [ ]:
region_pairs = discover_pairs(COMPOSITES_ROOT, MASKS_ROOT)

In [ ]:
# Train = Caucase | Val + Test = échantillonnage aléatoire des Alpes + Andes
train_pairs = region_pairs.get(TRAIN_REGION, [])

# Pooler Alpes + Andes et garder la trace de la région
eval_pool = []  # (pair, region)
for r in EVAL_REGIONS:
    for p in region_pairs.get(r, []):
        eval_pool.append((p, r))

rng = np.random.default_rng(SEED)
perm = rng.permutation(len(eval_pool))
n_val = max(1, int(len(eval_pool) * VAL_FRAC))

val_items  = [eval_pool[i] for i in perm[:n_val]]
test_items = [eval_pool[i] for i in perm[n_val:]]

val_pairs  = [item[0] for item in val_items]
test_pairs = [item[0] for item in test_items]

# Garder la région pour chaque paire (utile pour les métriques par région)
val_regions  = [item[1] for item in val_items]
test_regions = [item[1] for item in test_items]

splits = {
    "train": [(str(a), str(b)) for a, b in train_pairs],
    "val":   [(str(a), str(b)) for a, b in val_pairs],
    "test":  [(str(a), str(b)) for a, b in test_pairs],
    "val_regions":  val_regions,
    "test_regions": test_regions,
}
with open(SPLITS_PATH, "w") as f:
    json.dump(splits, f, indent=2) # on sauvegarde dans un json pour retrouver le dataset de test dans l'autre notebook
print(f"Splits sauvegardés dans {SPLITS_PATH}")

from collections import Counter
print(f"\nTrain : {len(train_pairs)} ({TRAIN_REGION})")
print(f"Val   : {len(val_pairs)}  (alps+andes mélangés)")
print(f"Test  : {len(test_pairs)}  (alps+andes mélangés)  ← NON UTILISÉ ICI")
print("\nDétail val :", Counter(val_regions))
print("Détail test:", Counter(test_regions))

In [ ]:
# Stats de normalisation (train uniquement)
mean, std = compute_band_stats(train_pairs, n_bands=N_BANDS)
print("Mean par bande:", mean)
print("Std  par bande:", std)

In [ ]:
train_ds = GlacierDataset(train_pairs, mean, std, augment=True,  pad_to=PAD_TO)
val_ds   = GlacierDataset(val_pairs,   mean, std, augment=False, pad_to=PAD_TO)

# Datasets de validation par région pour métriques séparées
val_pairs_by_region = {}
for (pair, reg) in val_items:
    val_pairs_by_region.setdefault(reg, []).append(pair)

val_datasets_by_region = {
    r: GlacierDataset(p, mean, std, augment=False, pad_to=PAD_TO)
    for r, p in val_pairs_by_region.items()
}

# num_workers=0 sur Windows
# HPC : num_workers=4, pin_memory=True
NUM_WORKERS = 0
PIN_MEMORY  = False

train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
val_loader   = DataLoader(val_ds,   BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

val_loaders_by_region = {
    r: DataLoader(ds, BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    for r, ds in val_datasets_by_region.items()
}

## Apperçu 

In [ ]:
import matplotlib.pyplot as plt

def show_samples(dataset, n=4):
    fig, axes = plt.subplots(n, 2, figsize=(6, 3 * n))
    if n == 1:
        axes = axes[np.newaxis]
    for i in range(min(n, len(dataset))):
        img, mask = dataset[i]
        rgb = img[[2, 1, 0]].numpy()
        for c in range(3):
            v = rgb[c][rgb[c] != 0]
            if len(v) > 0:
                lo, hi = np.percentile(v, [2, 98])
                rgb[c] = np.clip((rgb[c] - lo) / (hi - lo + 1e-6), 0, 1)
        axes[i, 0].imshow(rgb.transpose(1, 2, 0))
        axes[i, 0].set_title("Composite RGB")
        axes[i, 0].axis("off")
        axes[i, 1].imshow(mask[0].numpy(), cmap="gray", vmin=0, vmax=1)
        axes[i, 1].set_title("Masque GLIMS")
        axes[i, 1].axis("off")
    plt.tight_layout()
    plt.show()

show_samples(train_ds, n=4)

# Entrainement du modèle

In [ ]:
model = UNet(in_channels=N_BANDS).to(DEVICE)

# Vérification des dimensions
dummy = torch.randn(1, N_BANDS, PAD_TO, PAD_TO, device=DEVICE)
out = model(dummy)
print(f"Entrée  : {tuple(dummy.shape)}")
print(f"Sortie  : {tuple(out.shape)}")

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Paramètres entraînables : {n_params:,}")

In [ ]:
criterion = DiceBCELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

history = {"train_loss": [], "val_loss": [], "train_iou": [], "val_iou": [], "lr": []}
best_val_iou = 0.0

for epoch in range(1, EPOCHS + 1):
    train_loss, train_iou = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
    val_loss,   val_iou   = evaluate(model, val_loader, criterion, DEVICE)
    current_lr = optimizer.param_groups[0]["lr"]
    scheduler.step()

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_iou"].append(train_iou)
    history["val_iou"].append(val_iou)
    history["lr"].append(current_lr)

    if val_iou > best_val_iou:
        best_val_iou = val_iou
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_iou": val_iou,
            "mean": mean,
            "std": std,
        }, CHECKPOINT_DIR / "best_unet.pt")

    if epoch % 10 == 0 or epoch == 1:
        print(
            f"Epoch {epoch:3d}/{EPOCHS} | "
            f"train loss {train_loss:.4f}  IoU {train_iou:.3f} | "
            f"val loss {val_loss:.4f}  IoU {val_iou:.3f} | "
            f"lr {current_lr:.2e}"
        )

print(f"\nMeilleur IoU validation : {best_val_iou:.4f}")

In [ ]:
plot_history(history)

# Évaluation du modèle

In [ ]:
ckpt = torch.load(CHECKPOINT_DIR / "best_unet.pt", map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
print(f"Checkpoint chargé (epoch {ckpt['epoch']}, val IoU {ckpt['val_iou']:.4f})")

print("\n── Résultats validation globaux ──")
val_loss, val_iou = evaluate(model, val_loader, criterion, DEVICE)
print(f"  {'val (global)':20s} | loss {val_loss:.4f} | IoU {val_iou:.4f}")

print("\n── Résultats par région (validation) ──")
for region, loader in val_loaders_by_region.items():
    if len(loader.dataset) == 0:
        print(f"  {region:20s} | aucune donnée")
        continue
    r_loss, r_iou = evaluate(model, loader, criterion, DEVICE)
    print(f"  {region:20s} | loss {r_loss:.4f} | IoU {r_iou:.4f}")

In [ ]:
print("── Prédictions validation ──")
show_predictions(model, val_ds, DEVICE, n=3)

for region, ds in val_datasets_by_region.items():
    if len(ds) > 0:
        print(f"\n── Prédictions val ({region}) ──")
        show_predictions(model, ds, DEVICE, n=2)

## Sauvegarde du modèle comme ca on peut le récuperer après sans avoir a reentrainer
Le test set sera ensuite évalué lorsqu'on aura sauvegardé un modèle qui nous plait, on en fera un autre notebook

In [ ]:
torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "scheduler_state_dict": scheduler.state_dict(),
    "epoch": EPOCHS,
    "history": history,
    "best_val_iou": best_val_iou,
    "mean": mean,
    "std": std,
    "config": {
        "in_channels": N_BANDS,
        "features": [32, 64, 128, 256],
        "pad_to": PAD_TO,
        "batch_size": BATCH_SIZE,
        "lr": LR,
    },
}, CHECKPOINT_DIR / "unet_final.pt")

print("Modèle sauvegardé dans", CHECKPOINT_DIR / "unet_final.pt")